# Guide 12: Retrieval-augmented generation: answering from documents

*Lifecycle* · [Read the guide](https://drvgautam.github.io/primer-on-aiengineering/rag/) · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drvgautam/primer-on-aiengineering/blob/main/code/notebooks/12-rag.ipynb)

This notebook holds every code block from Guide 12, in the guide's order: 10 code cells, 10 of which print an output the guide quotes. Each cell is preceded by the guide paragraph that introduces it, so you can keep the guide open beside the notebook. The saved outputs are the guide's; `tools/verify_notebooks.py` checks they still match.

Run the setup cell first, then the cells in order: like the guide's blocks, they share variables.

**Sections of the guide**

- Answering from documents — no code
- The toy knowledge base — 1 code cell
- Sparse retrieval: matching words — 2 code cells
- Dense retrieval: matching learned vectors — 2 code cells
- Hybrid retrieval and reranking — 1 code cell
- The reader: answering from a passage — 2 code cells
- The whole pipeline, and how it fails — 1 code cell
- RAG, tools or fine-tuning? — no code
- The whole series in one pipeline — 1 code cell

In [1]:
# Setup: run this cell first. It makes the repository's code/ folder the working directory, so
# every block below runs exactly as printed in the guide ("run from the repository root").
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/drvgautam/primer-on-aiengineering"
if "google.colab" in sys.modules and not any((p / "code" / "notebooks").is_dir() for p in [Path.cwd(), *Path.cwd().parents]):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/primer-on-aiengineering"], check=True)
    os.chdir("/content/primer-on-aiengineering")
    pins = [l.strip() for l in open("code/requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins,
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], check=True)

here = Path.cwd().resolve()
code_dir = next(p / "code" for p in [here, *here.parents] if (p / "code" / "notebooks").is_dir())
os.chdir(code_dir)

# The guide's outputs were produced with these versions, on CPU. Others may differ in the last digits.
import numpy, tiktoken, torch
pins = dict(l.strip().split("==") for l in open("requirements.txt") if l.split("==")[0] in ("numpy", "tiktoken", "torch"))
for name, mod in (("numpy", numpy), ("tiktoken", tiktoken), ("torch", torch)):
    have = mod.__version__.split("+")[0]
    note = "" if have == pins[name] else f"  <- the guide used {pins[name]}; outputs may differ slightly (on Colab, restart the runtime and rerun this cell)"
    print(f"{name:9} {have}{note}")
print("working directory:", Path(code_dir.parent.name) / "code")

# %%gpt2 marks blocks the guide shows but doesn't run, because they need GPT-2 small's 548 MB weights.
# To run them, download the weights as series/running-example.py explains, then set:
#     os.environ["PRIMER_RUN_GPT2"] = "1"
from IPython.core.magic import register_cell_magic

@register_cell_magic
def gpt2(line, cell):
    if os.environ.get("PRIMER_RUN_GPT2") == "1":
        get_ipython().run_cell(cell)
    else:
        print("Skipped: needs GPT-2 weights (see the setup cell to enable).")


numpy     2.2.6
tiktoken  0.14.0
torch     2.14.1
working directory: primer-on-aiengineering/code


## The toy knowledge base

The examples use the series' toy world: sentences about six animals, where each animal has a home (cat hut, dog bed, rat log, fox rug, owl pot, hen mat), and Guide 8's base model, a 29,275-parameter character-level transformer pre-trained on those sentences. The knowledge base holds the six home sentences and 60 distractors drawn from Guide 7's sentence types, like "the fox sat by the hut". Run the blocks from the repository root, so Python can find the series' modules in `guides/`:

In [2]:
import sys
sys.path.insert(0, "guides")                   # run from the repository root
import math, random, torch
import torch.nn.functional as F
from rag import (kb, distractors, words, question, nouns, places, types, recall, ranks, top_passages,
                 skipgram, train_retriever, train_reader, reader_check, read)

print(len(kb), "passages;", sum(p.split()[1] == "cat" for p in distractors), "distractors are about the cat")
print(kb[0], "|", kb[6], "|", question("cat"))

66 passages; 10 distractors are about the cat
the cat lives in the hut | the fox sat by the hut | where does the cat live


## Sparse retrieval: matching words

The idea of weighting by rarity goes back to [Spärck Jones (1972)](https://doi.org/10.1108/eb026526). A related scoring function from search engines, **BM25** ("best matching", the 25th variant its authors tried), comes from a probabilistic model of relevance; it also weights rare words more, caps how much repeating a word can add, and adjusts for passage length ([Robertson & Zaragoza, 2009](https://doi.org/10.1561/1500000019)). To score a passage, compare its vector with the question's by cosine similarity. Recall from Guide 3 that $\cos(u, v) = \frac{u \cdot v}{\lVert u \rVert \, \lVert v \rVert}$, the dot product divided by both lengths, is 1 when two vectors point the same way and 0 when they share nothing. Vectors like these, mostly zeros, are called sparse, and so is retrieval that uses them:

In [3]:
df = {w: sum(w in p.split() for p in kb) for w in words}                # passages containing each word
idf = {w: math.log(len(kb) / df[w]) if df[w] else 0.0 for w in words}   # rarer words weigh more

def tfidf(text):                               # 27 numbers, one per word in `words`: count × idf
    v = torch.zeros(len(words))
    for w in text.split():
        v[words.index(w)] += idf[w]
    return v

def sparse(text, kb):                          # 66 scores: cosine of the text's vector with every passage's
    return F.cosine_similarity(tfidf(text)[None], torch.stack([tfidf(p) for p in kb]))

print(f"recall@1 {recall(sparse, kb, nouns, 1):.0%}, recall@3 {recall(sparse, kb, nouns, 3):.0%}")
print(top_passages(sparse, kb, question("cat")))
print({w: round(idf[w], 2) for w in ("the", "cat", "lives", "live", "where")})

recall@1 0%, recall@3 0%
['the cat hid by the hut', 'the cat hid on the rug', 'the cat lay on the bed']
{'the': 0.0, 'cat': 1.79, 'lives': 2.4, 'live': 0.0, 'where': 0.0}


That failure is called **vocabulary mismatch**: the question and the answer use different words for the same thing. To check the diagnosis, ask in the documents' own words:

In [4]:
rewritten = lambda a: f"the {a} lives in"      # the question, in the passages' words
print(f"rewritten: recall@1 {recall(sparse, kb, nouns, 1, ask=rewritten):.0%}")

rewritten: recall@1 100%


## Dense retrieval: matching learned vectors

**Dense retrieval** compares learned vectors of the question and each passage, pooled from word embeddings, instead of word counts; other sources call it embedding retrieval. Recall from Guide 3 that skip-gram learns a vector for each word by predicting its neighbors, so words used in similar contexts get similar vectors. Here skip-gram trains 8-number vectors on the text Guide 8's base model was pre-trained on. A whole text's vector is the average of its words' vectors, the pooling from Guide 3. As in Guide 3's toys, the retriever's tokens are whole words; real dense retrievers embed Guide 2's subword tokens and read them in context:

In [5]:
E = skipgram()                                 # |words| × 8: Guide 3's skip-gram on Guide 8's corpus

def embed(text, E):                            # a text's vector, 8 numbers: the average of its words' vectors
    return E[[words.index(w) for w in text.split()]].mean(0)

def dense(E):                                  # a retriever: 66 cosines of averaged vectors
    return lambda text, kb: F.cosine_similarity(embed(text, E)[None], torch.stack([embed(p, E) for p in kb]))

general = dense(E)
print(f"recall@1 {recall(general, kb, nouns, 1):.0%}, recall@3 {recall(general, kb, nouns, 3):.0%}")
print(top_passages(general, kb, question("cat")))
print({a: round(F.cosine_similarity(E[words.index("cat")], E[words.index(a)], dim=0).item(), 2) for a in nouns[1:]})   # cosine with "cat"

recall@1 0%, recall@3 0%
['the hen ran near the mat', 'the hen ran near the bed', 'the hen hid near the mat']
{'dog': 0.99, 'rat': 1.0, 'fox': 1.0, 'owl': 0.99, 'hen': 0.99}


The fix is to train the retriever for the task. **Contrastive training** teaches it to score the right passage above wrong ones. Here, for each question, take its right passage and 15 random others, and treat the 16 as options in a classification. Their cosines go through a softmax as logits, and Guide 1's cross-entropy loss, the negative log of the probability given to the correct option, pushes the right passage's probability up. Cosines only run from −1 to 1, which is too narrow: even with the right passage at 1 and the other 15 at −1, the softmax gives it only $e^{1}/(e^{1} + 15e^{-1}) \approx 0.33$. Dividing by a `sharpness` of 0.1 (papers call this divisor a temperature; it plays the same role as Guide 6's $T$ but is a separate setting) widens the range to −10 to 10, so the right passage's probability can approach 1. Training starts from the skip-gram vectors and adjusts them with Adam for 300 steps, one question per step, using questions about only three animals: the cat, the dog and the rat.

In [6]:
def contrastive_loss(q, candidates, sharpness=0.1):   # q: 8 numbers; candidates: 16 × 8, the right passage first
    scores = F.cosine_similarity(q[None], candidates) / sharpness
    return F.cross_entropy(scores[None], torch.tensor([0]))        # Guide 1's loss, with the right passage as class 0

E_trained = train_retriever(E, contrastive_loss, embed, seed=0)   # questions about cat, dog, rat only
trained = dense(E_trained)
for group in (nouns[:3], nouns[3:]):
    print(group, f"recall@1 {recall(trained, kb, group, 1):.0%}, recall@3 {recall(trained, kb, group, 3):.0%}")
print(top_passages(trained, kb, question("fox")))
pairs = [("cat", "dog"), ("cat", "rat"), ("fox", "owl"), ("fox", "hen")]
print({f"{a}-{b}": round(F.cosine_similarity(E_trained[words.index(a)], E_trained[words.index(b)], dim=0).item(), 2)
       for a, b in pairs})

['cat', 'dog', 'rat'] recall@1 100%, recall@3 100%
['fox', 'owl', 'hen'] recall@1 33%, recall@3 100%
['the hen lives in the mat', 'the owl lives in the pot', 'the fox lives in the rug']
{'cat-dog': 0.01, 'cat-rat': -0.63, 'fox-owl': 1.0, 'fox-hen': 0.99}


## Hybrid retrieval and reranking

The two kinds of retriever fail differently. Sparse retrieval is good at exact words, like names and codes, and blind to rewording. Dense retrieval matches whatever its training paired together, which can include rewordings that share no words, but it can blur names together, as it blurred the fox with the hen. **Hybrid retrieval** combines the two, here by adding half of each score:

In [7]:
def hybrid(text, kb, w_dense=0.5):            # half learned match, half word overlap
    return w_dense * trained(text, kb) + (1 - w_dense) * sparse(text, kb)

print(f"recall@1 on fox, owl, hen: {recall(hybrid, kb, nouns[3:], 1):.0%}; rank of each home sentence:",
      ranks(hybrid, kb, nouns))

recall@1 on fox, owl, hen: 100%; rank of each home sentence: [1, 1, 1, 1, 1, 1]


## The reader: answering from a passage

Two readers show what abstaining does. **Abstention** means answering "unknown" when the passage doesn't contain the answer. The first reader trains only on passages that answer. The second also gets passages that don't, 40% of its examples, with "unknown" as the response: half are another animal's home sentence, half a distractor about the same animal.

In [8]:
def reader_example(rng, abstain):             # one (prompt, response) pair, with a random place
    a, place = rng.choice(nouns), rng.choice(places)
    passage, answer = f"the {a} lives in the {place}", f"in the {place}"
    roll = rng.random()
    if abstain and roll < 0.2:                 # another animal's home sentence
        other = rng.choice([b for b in nouns if b != a])
        passage, answer = f"the {other} lives in the {place}", "unknown"
    elif abstain and roll < 0.4:               # a distractor about the same animal
        passage, answer = rng.choice([t for t in types if t.split()[1] == a]), "unknown"
    return f"{passage} {question(a)} ", f"{answer} and"

print(reader_example(random.Random(1), abstain=True))

('the dog lives in the pot where does the dog live ', 'in the pot and')


Each reader trains for 400 steps on batches of 32 drawn from 2,000 such examples, with Adam at a learning rate of 0.001. `reader_check` then tries 102 passages. There are 36 right home sentences and 36 distractors like "the cat sat on the mat", one of each per animal and place, and 30 pairings of an animal's question with a different animal's true home sentence. It counts the place copied correctly on the first, "unknown" or any place on the second, and "unknown" or the other animal's place on the third:

In [9]:
readers = {}
for abstain in (False, True):
    readers[abstain] = train_reader(reader_example, abstain, seed=0)    # 400 steps, about 10 s per reader
    print(f"abstain={abstain}:", reader_check(readers[abstain]))
    print("  given 'the cat lives in the pot':", read(readers[abstain], "the cat lives in the pot", "cat"))

abstain=False: {'copied': 36, 'distractor: unknown': 0, 'distractor: a place': 36, "other's home: unknown": 0, "other's home: its place": 30}
  given 'the cat lives in the pot': in the pot


abstain=True: {'copied': 36, 'distractor: unknown': 36, 'distractor: a place': 0, "other's home: unknown": 0, "other's home: its place": 30}
  given 'the cat lives in the pot': in the pot


## The whole pipeline, and how it fails

Now put the halves together and change the world: every animal moves. The cat now lives in the pot, the dog in the hut, the rat in the bed, the fox in the mat, the owl in the rug and the hen in the log. The update is an edit to six passages. No model is retrained. The retrievers are the seed-0 ones, and each one's top passage goes to the seed-0 reader trained with abstention from the last block (`readers[True]`):

In [10]:
moved = {"cat": "pot", "dog": "hut", "rat": "bed", "fox": "mat", "owl": "rug", "hen": "log"}
kb_moved = [f"the {a} lives in the {moved[a]}" for a in nouns] + distractors    # edit documents, retrain nothing
for name, score in [("TF-IDF", sparse), ("general", general), ("trained dense", trained), ("hybrid", hybrid)]:
    tally, misses = {"right": 0, "unknown": 0, "wrong": 0}, []
    for a in nouns:
        passage = top_passages(score, kb_moved, question(a), 1)[0]      # the top passage goes in the prompt
        answer = read(readers[True], passage, a)
        verdict = "right" if answer == f"in the {moved[a]}" else "unknown" if answer == "unknown" else "wrong"
        tally[verdict] += 1
        misses += [f"  {a}: {passage!r} -> {answer!r}"] * (verdict != "right")       # every miss, with its passage
    print(f"{name:13}  right {tally['right']}  unknown {tally['unknown']}  wrong {tally['wrong']}", *misses, sep="\n")

TF-IDF         right 0  unknown 6  wrong 0
  cat: 'the cat hid by the hut' -> 'unknown'
  dog: 'the dog lay near the bed' -> 'unknown'
  rat: 'the rat hid by the bed' -> 'unknown'
  fox: 'the fox hid by the bed' -> 'unknown'
  owl: 'the owl hid by the hut' -> 'unknown'
  hen: 'the hen hid by the bed' -> 'unknown'


general        right 0  unknown 5  wrong 1
  cat: 'the hen ran near the mat' -> 'in the mat'
  dog: 'the hen ran near the mat' -> 'unknown'
  rat: 'the hen hid by the bed' -> 'unknown'
  fox: 'the hen ran near the mat' -> 'unknown'
  owl: 'the hen hid by the bed' -> 'unknown'
  hen: 'the hen hid by the bed' -> 'unknown'


trained dense  right 3  unknown 0  wrong 3
  fox: 'the hen lives in the log' -> 'in the log'
  owl: 'the hen lives in the log' -> 'in the log'
  hen: 'the rat lives in the bed' -> 'in the bed'


hybrid         right 6  unknown 0  wrong 0


## The whole series in one pipeline

The series' running example, "The cat sat on the", makes one last appearance here, lowercased for the toy alphabet, as a search query (unrelated to attention's queries) to the toy knowledge base:

In [11]:
print(top_passages(sparse, kb, "the cat sat on the", 1), top_passages(trained, kb, "the cat sat on the", 1))

['the cat sat on the rug'] ['the cat lay by the hut']


---

← [Guide 11: Tool use and agents](11-tools.ipynb)

Found a cell that doesn't reproduce the guide? [Open an issue](https://github.com/drvgautam/primer-on-aiengineering/issues/new?template=does-not-reproduce.yml).